# Which vendors can I reach, and how far has each one been checked?

qb-compiler reaches several hardware vendors and several circuit formats. Each one is a tendril, and
each tendril declares the same things: what it supports, where its prices come from, where its
calibration comes from, and how far its numbers have been checked on real hardware.

This notebook lists them, runs the conformance check on two, shows the check failing a declaration
that claims more than it can show, and adds a tendril of our own without touching qb-compiler.

## 1. Every tendril, and its label

In [1]:
from datetime import date

from qb_compiler.tendrils import discover

found, broken = discover()
print(f"{'name':<12} {'kind':<9} {'label':<13} {'record':<8} covers to")
for t in found:
    print(f"{t.name:<12} {t.kind:<9} {t.label:<13} {t.validation_state():<8} {t.validated_until() or '-'}")
print("broken plugins:", len(broken))

name         kind      label         record   covers to
ibm          hardware  validated     expired  2026-06-12
ionq         hardware  UNVALIDATED   none     -
iqm          hardware  UNVALIDATED   none     -
quantinuum   hardware  UNVALIDATED   none     -
rigetti      hardware  UNVALIDATED   none     -
openqasm2    format    fixture-only  none     -
openqasm3    format    fixture-only  none     -
qiskit       format    fixture-only  none     -
cirq         format    fixture-only  none     -
pennylane    format    fixture-only  none     -
broken plugins: 0


The labels are the ones `qbc when` prints. Only IBM is `validated`, and it names the job it rests
on. That record is from March, so it no longer covers today: the listing says so instead of hiding
it.

In [2]:
ibm = next(t for t in found if t.name == "ibm")
print(ibm.validation)
print("record state today:", ibm.validation_state())
print("record state on 2026-05-01:", ibm.validation_state(date(2026, 5, 1)))

ValidationRecord(date='2026-03-14', backend='ibm_fez', job_id='d6qknvfr88ds73dch6r0', raw_sha256='ab898699c3fc00151c8e34dd8c2216f0595f19c25c81116d3e7b3c382581cf84', source='results/hardware_validation_dd.json in the qb-compiler repository, committed 2026-03-14', invoice_checked=False)
record state today: expired
record state on 2026-05-01: current


## 2. The conformance check

In [3]:
from qb_compiler.tendrils import check, verdict
from qb_compiler.tendrils.builtin import round_trip

for name in ("ibm", "openqasm2"):
    t = next(t for t in found if t.name == name)
    extra = [round_trip(name)] if t.kind == "format" else []
    checks = check(t, extra=extra)
    print(name, verdict(checks))
    for c in checks:
        print(f"   {str(c.passed):<5} {c.name:<18} {c.detail}")

ibm CONFORMS
   True  identity           complete
   True  capabilities       supports circuits, price, calibration
   True  label              validated on job d6qknvfr88ds73dch6r0
   True  price              3 rows, oldest read 2026-09-17
   True  calibration        live
   None  validation_window  record of 2026-03-14 is expired, covers to 2026-06-12
openqasm2 CONFORMS
   True  identity           complete
   True  capabilities       supports circuits
   True  label              fixture-only
   True  price              declared: no public price
   True  calibration        none
   None  validation_window  no dated validation record
   True  round_trip         GHZ-3 comes back unchanged


`None` marks a check that only reports. The OpenQASM 2 tendril also sends a three-qubit GHZ
circuit through the format and back and compares what returns.

## 3. A declaration that claims more than it shows

In [4]:
from dataclasses import replace

overclaim = replace(next(t for t in found if t.name == "ionq"), label="validated")
checks = check(overclaim)
print(verdict(checks))
print([(c.name, c.detail) for c in checks if c.passed is False])

DOES_NOT_CONFORM
[('label', 'validated, but no validation record is named')]


Calling IonQ `validated` without a record of a real job fails the label check by name. The same
rule applies to every tendril, ours included.

## 4. A tendril of your own

In [5]:
import tempfile
from pathlib import Path

from qb_compiler.signing import load_or_create_signing_key, sign_receipt, verify_receipt
from qb_compiler.tendrils import Tendril, conformance_receipt

mine = Tendril.from_dict({
    "name": "acme",
    "kind": "hardware",
    "vendor": "Acme Quantum",
    "maintainer": "Acme",
    "contact": "https://acme.example/support",
    "adapter_version": "1.0.0",
    "capabilities": {"circuits": True, "qec_results": False, "price": True,
                     "calibration": False, "execute": False},
    "label": "UNVALIDATED",
    "backends": ["acme_one"],
    "price_rows": [{"backend": "acme_one", "unit": "per_shot", "amount": 0.001,
                    "currency": "USD", "as_of": "2026-11-01",
                    "source": "https://acme.example/pricing"}],
})
checks = check(mine)
print("verdict:", verdict(checks))

with tempfile.TemporaryDirectory() as tmp:
    key = load_or_create_signing_key(Path(tmp) / "acme_key")
    receipt = sign_receipt(conformance_receipt(mine, checks), key=key)
    print("signed by:", receipt["key_fingerprint"] == key.fingerprint)
    print("verifies offline:", verify_receipt(receipt, public_key=key.public).status)
    receipt["verdict"] = "CONFORMS, honest"
    print("after tampering:", verify_receipt(receipt, public_key=key.public).status)

verdict: CONFORMS


signed by: True
verifies offline: VERIFIED
after tampering: INVALID_SIGNATURE


In a package, the same declaration goes behind an entry point and `qbc tendrils` lists it:

```toml
[project.entry-points."qb_compiler.tendrils"]
acme = "acme_qbc:TENDRIL"
```

The receipt is signed with the submitter's key, not ours, so a vendor can publish one for its own
tendril. See `docs/tendrils.md`.

## What this does not claim

The check is about the declaration. It does not say the hardware is any good, and a price is only
as current as the date it was read.